# Tutorial: From Prompts to Data

## AI as a Measurement Tool in Behavioral Data Science

In the previous module, you learned how to communicate with a large language model using Python and an API key. In this tutorial, you will take the next step: you will learn how to use an AI system to create a small structured dataset from text.

In many professional settings, AI is not used simply as a chatbot. Instead, it is used to analyze, classify, summarize, and label large collections of text. This happens in behavioral science, education, product design, public health, and UX/UI work. For example, a company may collect thousands of user comments about an app and use AI to classify whether each comment expresses satisfaction, confusion, frustration, or a request for a new feature. Once those responses are coded into categories, they become data that can be counted, plotted, and analyzed.

That is the central idea of this tutorial. We will move from **conversation** to **measurement**.

By the end of this tutorial, you will be able to:

1. Send multiple prompts to a language model using Python.
2. Ask the model for responses in a structured format.
3. Convert those responses into a Pandas DataFrame.
4. Perform simple descriptive analyses on the resulting dataset.
5. Reflect on how prompt design changes what the model measures.

Throughout this notebook, remember the conceptual analogy:

- **Prompt** = stimulus
- **Model** = participant
- **Response** = output
- **Dataset** = collection of outputs across many trials

Let us begin.

## 0. Before You Begin: Environment Setup

In Python, packages such as `openai`, `pandas`, and `matplotlib` are installed within specific environments.

Sometimes, Jupyter notebooks run in a different environment than the one where packages were installed. When that happens, you may see an error such as:

`ModuleNotFoundError: No module named 'openai'`

The next cell will check whether the required packages are installed and install them if needed. After that, a diagnostic cell will show which Python executable your notebook is currently using.

If something does not work at first, do not panic. This is one of the most common setup issues in data science, even for experienced programmers.

In [ ]:
# =========================================
# Setup Check: Python Environment & Packages
# =========================================

import sys
import subprocess
import importlib

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        importlib.import_module(import_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} not found. Installing now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

required_packages = [
    ("openai", "openai"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib")
]

for package_name, import_name in required_packages:
    install_if_missing(package_name, import_name)

print("\nAll required packages are ready.")

In [ ]:
# =========================================
# Environment Diagnostic
# =========================================

import sys

print("Python executable being used by this notebook:")
print(sys.executable)

print("\nIf this path does not include your expected conda environment,")
print("you may be running the wrong Jupyter kernel.")

## 1. Import the packages we will use

We will use:

- `os` to access the API key stored on your computer
- `json` to work with structured responses
- `pandas` to create a table of results
- `matplotlib.pyplot` to make a simple plot
- `time` to briefly pause between API calls if needed
- `OpenAI` to communicate with the model

In [ ]:
import os
import json
import time
import pandas as pd
import matplotlib.pyplot as plt

from openai import OpenAI

## 2. Connect to the API

As in the previous module, we will connect to the model using an API key. The safest way to do this is to store the key as an environment variable named `OPENAI_API_KEY`.

The cell below looks for that key on your computer and uses it to create a client.

If you get an error here, revisit the setup from the previous module before continuing.

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError(
        "OPENAI_API_KEY was not found. Make sure your API key is stored as an environment variable."
    )

client = OpenAI(api_key=api_key)
print("Client created successfully.")

## 3. A first example: one sentence, one response

We will begin with a simple behavioral science task. Imagine that we have short open-ended responses from people describing how they feel. We want the model to identify the emotional tone of each response.

In [ ]:
text = "I feel nervous every time I have to speak in front of the class."
print(text)

In [ ]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=f"What is the emotional tone of this sentence? Sentence: {text}"
)

reply = response.output_text
print(reply)

### Reflection

The response is informative, but it is not yet ideal for data analysis. The model may respond using a full sentence, a paragraph, or different wording each time. That makes it difficult to count responses or compare them across many observations.

To use AI in a professional workflow, we usually need the output to be more structured.

## 4. Why structure matters

Suppose we want the model to return:

- an emotion category
- a valence label
- an intensity score from 1 to 5

If the model returns these values consistently, then we can store them in a table and analyze them with Python.

Let us try to request a structured answer using JSON.

### What is JSON?

**JSON** stands for **JavaScript Object Notation**. Despite the name, it is widely used far beyond JavaScript. It is a lightweight text format for storing information as **key-value pairs**.

A JSON object looks like this:

```json
{"emotion": "anxiety", "valence": "negative", "intensity": 4}
```

This is useful because:

- humans can read it fairly easily
- computers can parse it reliably
- the same fields can be repeated across many observations

In this tutorial, the model will return a short JSON object as text. Then we will use Python's `json.loads()` function to convert that text into a Python dictionary.

The word **loads** means "load from a string." In other words, Python reads the JSON-formatted text and turns it into a data structure we can work with directly.

In [ ]:
structured_prompt = f'''
Read the sentence below and classify it.

Sentence: "{text}"

Return your answer in valid JSON with exactly these keys:
- "emotion"
- "valence"
- "intensity"

Rules:
- "emotion" should be a single short category such as anxiety, joy, sadness, anger, stress, or neutral.
- "valence" should be one of: positive, negative, neutral.
- "intensity" should be an integer from 1 to 5.
- Return JSON only. Do not include any extra text.
'''

response = client.responses.create(
    model="gpt-4.1-mini",
    input=structured_prompt
)

json_text = response.output_text
print("Raw model output:")
print(json_text)

In [ ]:
result = None

try:
    result = json.loads(json_text)
except json.JSONDecodeError:
    print("Warning: JSON parsing failed on the first attempt.")

    try:
        start = json_text.find("{")
        end = json_text.rfind("}") + 1
        cleaned_text = json_text[start:end]
        result = json.loads(cleaned_text)
        print("Recovered JSON by extracting the text between braces.")
    except Exception:
        print("Could not recover valid JSON from the output.")

result

Excellent. We now have something that behaves like data rather than free-form prose.

This step is important conceptually: in professional work, AI outputs should be **validated** before they are treated as data.

## 5. Mini-Exercises: Practice Before Scaling Up

Before we build a dataset, let us pause and practice the basic ideas. These short exercises are designed to help you gain fluency. The first exercises ask you to **modify existing code**. The later exercises ask you to **write more on your own**.

Work through them in order.

### Mini-Exercise 1: Change the sentence

In the next cell, the code from the first example has been copied for you. Change the sentence so that it expresses a clearly **positive** emotional state, then run the cell and inspect the response.

In [ ]:
# One possible answer

text_ex1 = "I feel excited and confident about presenting my project tomorrow."

response_ex1 = client.responses.create(
    model="gpt-4.1-mini",
    input=f"What is the emotional tone of this sentence? Sentence: {text_ex1}"
)

print(response_ex1.output_text)

### Mini-Exercise 2: Change the prompt

Below, the model is asked for JSON output. Modify the prompt so that the model returns the same three variables but uses a **smaller emotion vocabulary**:

- positive
- negative
- neutral

Then run the cell and inspect the raw output.

In [ ]:
# One possible answer

text_ex2 = "I am relieved that the exam is finally over."

prompt_ex2 = f'''
Read the sentence below and classify it.

Sentence: "{text_ex2}"

Return your answer in valid JSON with exactly these keys:
- "emotion"
- "valence"
- "intensity"

Rules:
- "emotion" must be exactly one of: positive, negative, neutral.
- "valence" should be one of: positive, negative, neutral.
- "intensity" should be an integer from 1 to 5.
- Return JSON only. Do not include any extra text.
'''

response_ex2 = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt_ex2
)

print(response_ex2.output_text)

### Mini-Exercise 3: Parse the JSON

In the next cell, some code is already written. Complete it so that the raw output is safely converted into a Python dictionary. If parsing fails, try extracting the text between the first `{` and the last `}`.

In [ ]:
# One possible answer

raw_output_ex3 = response_ex2.output_text
parsed_ex3 = None

try:
    parsed_ex3 = json.loads(raw_output_ex3)
except json.JSONDecodeError:
    print("First parsing attempt failed.")

    try:
        start = raw_output_ex3.find("{")
        end = raw_output_ex3.rfind("}") + 1
        cleaned_ex3 = raw_output_ex3[start:end]
        parsed_ex3 = json.loads(cleaned_ex3)
    except Exception:
        print("Could not recover JSON.")

parsed_ex3

### Mini-Exercise 4: Modify a function

Below is a function template. Modify it so that it also asks the model to return a fourth field called `"confidence"` with an integer from 1 to 5.

In [ ]:
def code_emotion_ex4(text, model="gpt-4.1-mini"):
    prompt = f'''
    Read the sentence below and classify it.

    Sentence: "{text}"

    Return your answer in valid JSON with exactly these keys:
    - "emotion"
    - "valence"
    - "intensity"
    - "confidence"

    Rules:
    - "emotion" should be a single short category.
    - "valence" should be one of: positive, negative, neutral.
    - "intensity" should be an integer from 1 to 5.
    - "confidence" should be an integer from 1 to 5.
    - Return JSON only. Do not include any extra text.
    '''

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text

code_emotion_ex4("I feel optimistic about this week.")

The next exercises ask you to write more code yourself.

### Mini-Exercise 5: Write a prompt from scratch

Write code that sends the sentence below to the model and asks for valid JSON with exactly these keys:

- `"emotion"`
- `"valence"`
- `"intensity"`

Use the sentence:

`"I feel disappointed that my hard work did not pay off."`

Print the raw model output.

In [ ]:
text_ex5 = "I feel disappointed that my hard work did not pay off."

prompt_ex5 = f'''
Read the sentence below and classify it.

Sentence: "{text_ex5}"

Return your answer in valid JSON with exactly these keys:
- "emotion"
- "valence"
- "intensity"

Rules:
- "emotion" should be a single short category.
- "valence" should be one of: positive, negative, neutral.
- "intensity" should be an integer from 1 to 5.
- Return JSON only. Do not include any extra text.
'''

response_ex5 = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt_ex5
)

print(response_ex5.output_text)

### Mini-Exercise 6: Build a small list of coded results

Create a list with **two short responses of your own**, then use a loop and the model to classify them into JSON outputs. Save the results in a list called `mini_results`.

In [ ]:
my_responses_ex6 = [
    "I feel relaxed after finishing my work.",
    "I am worried that I forgot something important."
]

mini_results = []

for item in my_responses_ex6:
    prompt_ex6 = f'''
    Read the sentence below and classify it.

    Sentence: "{item}"

    Return your answer in valid JSON with exactly these keys:
    - "emotion"
    - "valence"
    - "intensity"

    Rules:
    - "emotion" should be a single short category.
    - "valence" should be one of: positive, negative, neutral.
    - "intensity" should be an integer from 1 to 5.
    - Return JSON only. Do not include any extra text.
    '''

    response_ex6 = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt_ex6
    )

    raw_ex6 = response_ex6.output_text
    try:
        parsed_ex6 = json.loads(raw_ex6)
    except json.JSONDecodeError:
        start = raw_ex6.find("{")
        end = raw_ex6.rfind("}") + 1
        parsed_ex6 = json.loads(raw_ex6[start:end])

    parsed_ex6["response_text"] = item
    mini_results.append(parsed_ex6)
    time.sleep(1)

mini_results

### Mini-Exercise 7: Turn your results into a DataFrame

Using the `mini_results` list from the previous exercise, create a Pandas DataFrame and display it.

In [ ]:
mini_df = pd.DataFrame(mini_results)
mini_df

## 6. From one sentence to many sentences

A real data science workflow usually involves many observations. Below is a small synthetic dataset of short responses. These are not real participant data; they are simple examples created for teaching.

To keep the tutorial efficient and reduce the chance of rate-limit errors, we will work with a small set of responses.

In [ ]:
responses = [
    "I am excited to start my new internship next week.",
    "I feel overwhelmed by everything I need to finish.",
    "I do not really care one way or the other.",
    "I am proud that I finally solved the problem.",
    "I feel tired and unmotivated lately."
]

pd.DataFrame({"response_text": responses})

## 7. Write a reusable function

Instead of copying the same API call over and over, we will write a Python function. A function lets us define a set of instructions once and then reuse it for each response.

In professional work, we also need to make functions **robust**. Sometimes the model returns text that is almost JSON, but not perfectly valid JSON. Sometimes an API call fails temporarily. Good data pipelines should anticipate these possibilities.

The function below includes simple error handling for both cases.

In [ ]:
def code_emotion(text, model="gpt-4.1-mini"):
    prompt = f'''
    Read the sentence below and classify it.

    Sentence: "{text}"

    Return your answer in valid JSON with exactly these keys:
    - "emotion"
    - "valence"
    - "intensity"

    Rules:
    - "emotion" should be a single short category such as anxiety, joy, sadness, anger, stress, frustration, confusion, calm, pride, or neutral.
    - "valence" should be one of: positive, negative, neutral.
    - "intensity" should be an integer from 1 to 5.
    - Return JSON only. Do not include any extra text before or after the JSON object.
    '''

    output_text = None

    try:
        response = client.responses.create(
            model=model,
            input=prompt
        )

        output_text = response.output_text.strip()

        return json.loads(output_text)

    except json.JSONDecodeError:
        print("Warning: JSON parsing failed. Raw output was:")
        print(output_text)

        try:
            start = output_text.find("{")
            end = output_text.rfind("}") + 1
            cleaned_text = output_text[start:end]
            return json.loads(cleaned_text)
        except Exception:
            print("Warning: Could not recover valid JSON.")
            return {"emotion": None, "valence": None, "intensity": None}

    except Exception as e:
        print("Warning: API call failed.")
        print(e)
        time.sleep(2)
        return {"emotion": None, "valence": None, "intensity": None}

In [ ]:
code_emotion("I feel nervous when I have to talk in front of a large group.")

### Why this extra code matters

This may look slightly more complicated than our earlier examples, but it reflects a real principle of professional data work:

> AI outputs should be validated before they are treated as data.

Even if the model usually follows instructions, a good workflow checks whether the output is actually usable.

## 8. Build a dataset

Now we will use a `for` loop to apply the function to every response in our list. For each item, we will save:

- the original text
- the emotion category
- the valence
- the intensity

This will create a list of dictionaries that we can easily convert into a DataFrame.

To reduce the chance of rate-limit errors, we will pause briefly between calls.

In [ ]:
coded_results = []

for text in responses:
    coded = code_emotion(text)
    coded["response_text"] = text
    coded_results.append(coded)
    time.sleep(1)

coded_results[:3]

In [ ]:
df = pd.DataFrame(coded_results)
df

At this point, you have created a small AI-powered behavioral dataset.

## 9. Inspect the dataset

Before doing analysis, it is always good practice to inspect the data.

Look at:

- the first few rows
- the column names
- the data types
- whether any values are missing

In [ ]:
print(df.head())
print("\nColumn names:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

## 10. Descriptive analysis

Now that the AI outputs are stored as data, we can analyze them.

Let us begin with two simple questions:

1. Which emotion categories appear most often?
2. What is the average intensity score?

In [ ]:
emotion_counts = df["emotion"].value_counts()
emotion_counts

In [ ]:
average_intensity = df["intensity"].mean()
print(f"Average intensity: {average_intensity:.2f}")

In [ ]:
emotion_counts.plot(kind="bar")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.title("Emotion Categories Assigned by the Model")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Prompt design is measurement design

The model's outputs depend on how we ask the question. This means that prompt design is part of the measurement process.

Let us compare two prompt styles:

- a **vague** prompt
- a **structured** prompt

In [ ]:
comparison_text = "I cannot stop thinking about everything that might go wrong."
print(comparison_text)

In [ ]:
vague_response = client.responses.create(
    model="gpt-4.1-mini",
    input=f"What is the emotional tone of this sentence? Sentence: {comparison_text}"
)

print("Vague prompt output:\n")
print(vague_response.output_text)

In [ ]:
structured_response = client.responses.create(
    model="gpt-4.1-mini",
    input=f'''
    Read the sentence below and classify it.

    Sentence: "{comparison_text}"

    Return your answer in valid JSON with exactly these keys:
    - "emotion"
    - "valence"
    - "intensity"

    Rules:
    - "emotion" should be a single short category.
    - "valence" should be one of: positive, negative, neutral.
    - "intensity" should be an integer from 1 to 5.
    - Return JSON only. Do not include any extra text before or after the JSON object.
    '''
)

print("Structured prompt output:\n")
print(structured_response.output_text)

### Reflection

Notice the difference:

- The vague prompt often produces rich language that is useful for reading.
- The structured prompt produces compact output that is useful for analysis.

Neither is inherently better. They serve different goals. In professional work, the important question is: **What kind of output do you need for the task at hand?**

## 11A. From categories to richer measurement

In Section 11, we compared a vague prompt with a structured prompt for a **single sentence**. That comparison shows that prompt design affects what kind of output we obtain.

We can now take that idea one step further. Instead of assigning only an emotion category and an intensity score, we can ask the model to rate several emotional dimensions for each response. This creates a richer measurement profile.

For example, rather than asking only for one label such as `"anxiety"` or `"joy"`, we can ask the model to assign scores from 1 to 5 for multiple emotional dimensions such as:

- anxiety
- joy
- frustration
- calm

This is often closer to how measurement works in behavioral science. Many psychological constructs are not all-or-none categories. Instead, they vary in degree.

In professional settings, this kind of richer scoring can be useful for:

- analyzing user feedback in UX/UI research
- measuring multiple emotional reactions to text
- comparing responses across participants or conditions
- creating features for later statistical or machine learning models

### A new function for multi-emotion ratings

The function below asks the model to rate four emotional dimensions for a single response. Notice that the output is still JSON, but the structure is a little richer than before.

In [ ]:
def rate_multiple_emotions(text, model="gpt-4.1-mini"):
    prompt = f'''
    Read the sentence below and rate the extent to which it expresses each of the following emotions.

    Sentence: "{text}"

    Return your answer in valid JSON with exactly these keys:
    - "anxiety"
    - "joy"
    - "frustration"
    - "calm"

    Rules:
    - Each value should be an integer from 1 to 5.
    - 1 means the emotion is barely present.
    - 5 means the emotion is strongly present.
    - Return JSON only. Do not include any extra text before or after the JSON object.
    '''

    output_text = None

    try:
        response = client.responses.create(
            model=model,
            input=prompt
        )

        output_text = response.output_text.strip()
        return json.loads(output_text)

    except json.JSONDecodeError:
        print("Warning: JSON parsing failed. Raw output was:")
        print(output_text)

        try:
            start = output_text.find("{")
            end = output_text.rfind("}") + 1
            cleaned_text = output_text[start:end]
            return json.loads(cleaned_text)
        except Exception:
            print("Warning: Could not recover valid JSON.")
            return {"anxiety": None, "joy": None, "frustration": None, "calm": None}

    except Exception as e:
        print("Warning: API call failed.")
        print(e)
        time.sleep(2)
        return {"anxiety": None, "joy": None, "frustration": None, "calm": None}

Let us test the function on one sentence.

In [ ]:
rate_multiple_emotions("I feel overwhelmed and worried about everything I still need to do.")

Now let us apply this richer measurement approach to **multiple responses**.

In [ ]:
emotion_profiles = []

for text in responses:
    ratings = rate_multiple_emotions(text)
    ratings["response_text"] = text
    emotion_profiles.append(ratings)
    time.sleep(1)

emotion_profile_df = pd.DataFrame(emotion_profiles)
emotion_profile_df

We can now summarize the average score for each emotional dimension across all responses.

In [ ]:
average_profiles = emotion_profile_df[["anxiety", "joy", "frustration", "calm"]].mean()
average_profiles

Unlike the plot in Section 10, this plot shows emotional dimensions that can take **different average values**. This gives us a more nuanced picture of the dataset.

In [ ]:
average_profiles.plot(kind="bar")
plt.xlabel("Emotion Dimension")
plt.ylabel("Average Score")
plt.title("Average Multi-Emotion Ratings Across Responses")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interpretation

This plot is conceptually different from the count plot in Section 10.

- In Section 10, each response received **one category**, so the plot showed how often each category appeared.
- Here, each response receives **multiple ratings**, so the plot shows the **average strength** of several emotional dimensions.

This is often a more realistic representation of human language. A single sentence may express more than one emotional tone at the same time.

### Mini-Exercise 8: Modify the emotion dimensions

In the next cell, modify the function so that instead of rating:

- anxiety
- joy
- frustration
- calm

it rates these four dimensions:

- stress
- optimism
- sadness
- confidence

Then test the function on one sentence of your choice.

In [ ]:
def rate_multiple_emotions_ex8(text, model="gpt-4.1-mini"):
    prompt = f'''
    Read the sentence below and rate the extent to which it expresses each of the following emotions.

    Sentence: "{text}"

    Return your answer in valid JSON with exactly these keys:
    - "stress"
    - "optimism"
    - "sadness"
    - "confidence"

    Rules:
    - Each value should be an integer from 1 to 5.
    - Return JSON only. Do not include any extra text before or after the JSON object.
    '''

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text

# Test your modified function here
rate_multiple_emotions_ex8("I feel hopeful that my preparation will pay off.")

### Mini-Exercise 9: Create a new summary plot

Use the `emotion_profile_df` DataFrame from this section to create a new plot showing the **scores for one single response** across the four emotion dimensions.

Choose either:

- the first row of the dataset, or
- a row of your choice

This is useful because it lets you compare the emotional profile of one response rather than the average across all responses.

In [ ]:
single_profile = emotion_profile_df.loc[0, ["anxiety", "joy", "frustration", "calm"]]

single_profile.plot(kind="bar")
plt.xlabel("Emotion Dimension")
plt.ylabel("Score")
plt.title("Emotion Profile for One Response")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. A note on validity and reliability

In behavioral science, a good measurement tool must be valid and reliable.

- **Validity** asks whether the tool is really measuring what we think it is measuring.
- **Reliability** asks whether it produces consistent results.

These questions also apply to AI-based coding.

## 13. Common issues and what to do

### If you see `RateLimitError`
This usually means your account has hit a usage limit or too many requests were sent too quickly.

Try:
- waiting 10 to 20 seconds and running the cell again
- reducing the number of responses
- checking whether your API account has available credits

### If you see `JSONDecodeError`
This usually means the model returned text that was not perfectly valid JSON.

Try:
- printing the raw output to inspect it
- checking whether there was extra text before or after the JSON object
- using the recovery code in the function above

## 14. Summary

In this tutorial, you learned how to:

- turn free text into structured AI outputs
- understand what JSON is and load it into Python
- parse JSON in Python
- build a Pandas DataFrame from model responses
- compute descriptive summaries of AI-generated labels
- think about prompts as measurement tools
- handle a few common real-world errors

You also completed a set of short exercises that moved from **modifying existing code** to **writing small pieces of code independently**.